# Building Fast Queries on a CSV
## Reading the Dataset
Use the `csv` module to read the `laptops.csv` file and separate the header from the rows:

In [1]:
import csv

In [2]:
with open('laptops.csv', encoding='utf-8') as file:
    read_file = list(csv.reader(file))
    header = read_file[0]
    rows = read_file[1:]

print(header)
for i in range(5):
    print(rows[i])

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
['6571244', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 2.3GHz', '8GB', '128GB SSD', 'Intel Iris Plus Graphics 640', 'macOS', '1.37kg', '1339']
['7287764', 'Apple', 'Macbook Air', 'Ultrabook', '13.3', '1440x900', 'Intel Core i5 1.8GHz', '8GB', '128GB Flash Storage', 'Intel HD Graphics 6000', 'macOS', '1.34kg', '898']
['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', '575']
['9722156', 'Apple', 'MacBook Pro', 'Ultrabook', '15.4', 'IPS Panel Retina Display 2880x1800', 'Intel Core i7 2.7GHz', '16GB', '512GB SSD', 'AMD Radeon Pro 455', 'macOS', '1.83kg', '2537']
['8550527', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 3.1GHz', '8GB', '256GB SSD',

## Create an Inventory Сlass
Start implementing a class to represent the inventory. It get the name of the CSV file as argument and reads it into self.header and self.rows:

In [3]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])  # Convert the data in the "Price" column from strings to integers

In [4]:
# Let's check our class and print the headers and number of rows in the dataset
inventory = Inventory('laptops.csv')
print(inventory.header)
print(f'Number of rows: {len(inventory.rows)}')

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
Number of rows: 1303


## Finding a Laptop From the Id
Implement a `get_laptop_from_id()` method that given a laptop identifier find the row corresponding to that laptop:

In [5]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])

    def get_laptop_from_id(self, laptop_id):  # step 1
        for row in self.rows:                 # step 2
            if row[0] == laptop_id:
                return row
        return None                           # step 3

In [6]:
# let's check our function in action
inventory = Inventory('laptops.csv')
print(inventory.get_laptop_from_id('3362737'))
print(inventory.get_laptop_from_id('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


## Improving Id Lookups
Improve the time complexity of finding a laptop with a given ID by precomputing a dictionary that maps laptop IDs to strings.</br>
To improve the time complexity of finding a laptop with a given ID:
* I'll create a dictionary `id_to_row` in which I will add the keys - the laptop ID, and the values - the list with data for this laptop
* I'll create a new method called `get_laptop_from_id_fast()`, which will look up the laptop by its id in the dictionary

In [7]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])
        self.id_to_row = {}                        # step 1
        for row in self.rows:                      # step 2
            self.id_to_row[row[0]] = row

    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:
            if row[0] == laptop_id:
                return row
        return None

    def get_laptop_from_id_fast(self, laptop_id):  # step 3
        if laptop_id in self.id_to_row:            # step 4
            return self.id_to_row[laptop_id]
        return None

In [8]:
# let's test the code
inventory = Inventory('laptops.csv')
print(inventory.get_laptop_from_id('3362737'))
print(inventory.get_laptop_from_id('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


## Comparing the Performance
I'll compare the performance of the methods `get_laptop_from_id`, which performs a search in a list, and `get_laptop_from_id_fast`, which performs a search in pre-prepared data - in a dictionary `id_to_row`.</br>
To measure the execution time of methods, I import the module `time`. To generate pseudo-random values, I import the module `random`.

In [9]:
import time
import random

In [10]:
# I'll create a list with 10,000 values for laptop IDs
ids = [str(random.randint(1_000_000, 9_999_999)) for _ in range(10_000)]

inventory = Inventory('laptops.csv')
total_time_no_dict = 0
for value in ids:
    start = time.time()
    inventory.get_laptop_from_id(value)
    end = time.time()
    total_time_no_dict += end - start

total_time_dict = 0
for value in ids:
    start = time.time()
    inventory.get_laptop_from_id_fast(value)
    end = time.time()
    total_time_dict += end - start

print(f'Total list search execution time: {total_time_no_dict}')
print(f'Total dictionary search execution time: {total_time_dict}')

Total list search execution time: 0.6686553955078125
Total dictionary search execution time: 0.002818584442138672


### Analysis
We got:
* 0.6686553955078125
* 0.002818584442138672

We can see a significant improve in performance. If we divide `0.698` by `0.002` we see that the new method is about `334` times faster for this input size.